In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split, GroupShuffleSplit

import warnings as wr
wr.filterwarnings('ignore')

np.random.seed(1)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Read Test Ground Truth CSV File

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/ML Project/HAM10000_metadata.csv")
print(df)
print(df.shape)

In [ ]:
df.describe()

# Clean the Dataset

In [ ]:
missing = df.isna().sum()
print(missing)

In [ ]:
df[df.isna().any(axis=1)]

In [ ]:
df = df.dropna(subset=["age"]).reset_index(drop=True)

# Visualizing the Data

In [ ]:
counts = df["dx"].value_counts()

plt.figure(figsize=(9, 5))
bars = plt.bar(counts.index, counts.values)
plt.bar_label(bars)

plt.title("Number of Images per Lesion Type")
plt.ylabel("Number of Images")
plt.xlabel("Lesion Type")

plt.show()

In [ ]:
malignant_types = ["mel", "bcc"]
malignant = df[df["dx"].isin(malignant_types)]
benign = df[~df["dx"].isin(malignant_types)]

fig, ax = plt.subplots(1, 2, figsize=(20, 7))

locations = sorted(df['localization'].dropna().unique())

sns.histplot(data=df, x='age', hue='localization', hue_order=locations, multiple='stack', ax=ax[0])
sns.move_legend(ax[0], loc='upper left', title='Location', title_fontsize=13, fontsize=12)
ax[0].set_title('Age Histogram Localization Area Wise', fontsize=20)
ax[0].set_xlabel('Age', fontsize=14)
ax[0].set_ylabel('Frequency of Occurrence', fontsize=14)

ax[1].hist(malignant["age"].dropna(), bins=30, label="Malignant", alpha=0.5)
ax[1].hist(benign["age"].dropna(), bins=30, label="Benign", alpha=0.5)
ax[1].set_title('Distribution of Age by Diagnosis', fontsize=20)
ax[1].set_xlabel('Age', fontsize=14)
ax[1].set_ylabel('Count', fontsize=14)
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
per_lesion = df.groupby("lesion_id").size().value_counts().sort_index()

bars = plt.bar(per_lesion.index.astype(str), per_lesion.values)
plt.bar_label(bars)
plt.title("How Many Images Each Lesion Has")
plt.xlabel("Images per lesion")
plt.ylabel("Number of lesions")
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
sns.catplot(y="dx", x="age", hue="sex",kind="box", height = 6, aspect = 2.5,  data=df).set(title='Relationship between the type of lesion, age and the patient sex')
plt.ylabel('Lesion Type')
plt.xlabel('Age')
plt.show()

# Pre-process the data

In [ ]:
names = ['Melanocytic nevi','Melanoma','Benign keratosis-like lesions ',
               'Basal cell carcinoma','Actinic keratoses','Vascular lesions',
               'Dermatofibroma']

lesion_img = df.groupby('lesion_id')['image_id']\
               .count()\
               .to_frame()

lesion_dict = {
    'nv':'Melanocytic Nevi',
    'mel': 'Melanoma',
    'bkl': 'Benign keratosis',
    'bcc': 'Basal cell carcinoma',
    'akiec': 'Actinic keratoses',
    'vasc': 'Vascular lesions',
    'df': 'Dermatofibroma'

}

lesion_ID = {
    'nv': 0,
    'mel': 1,
    'bkl': 2,
    'bcc': 3,
    'akiec': 4,
    'vasc': 5,
    'df': 6
}

df['cell_type'] = df['dx'].map(lesion_dict.get)
df['lesion_ID'] = df['dx'].map(lesion_ID.get)

In [ ]:
path = '/content/drive/MyDrive/HAM10000_images'

In [ ]:
images_path = {os.path.splitext(os.path.basename(x))[0]:
               x for x in glob.glob('/content/drive/MyDrive/HAM10000_images/*.jpg')}

In [ ]:
df['path'] = df['image_id'].map(images_path.get)

In [ ]:
df

In [ ]:
df["path"].head()

In [ ]:
num_unique_id = df["image_id"].nunique()
num_unique_id

In [ ]:
# CHANGED: 71 -> 128. The source images are 600x450; 71x71 discards ~98% of the
# pixels, and with them the pigment networks, dots and streaks that the diagnosis
# actually depends on. This single line was the largest source of lost accuracy.
IMG_SIZE = 128

def get_img(img_path):
  img = cv2.imread(img_path,1)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # OpenCV loads BGR; everything downstream expects RGB
  img = cv2.resize(img,(IMG_SIZE, IMG_SIZE))
  return img


In [ ]:
from tqdm import tqdm

x = []
for img_name in tqdm(df['path'].values):
  x.append(get_img(img_name))

In [ ]:
# x is a Python list of 9958 separate arrays. Stack it into one array so rows can
# be selected by index - that is what the split below returns.
x = np.array(x, dtype=np.uint8)
y = df["lesion_ID"].values
groups = df["lesion_id"].values      # which physical lesion each image came from

print("x:", x.shape, x.dtype)
print("y:", y.shape, y.dtype)
print("groups:", groups.shape, "|", len(np.unique(groups)), "unique lesions")

In [ ]:
from numpy.core.function_base import linspace

plt.figure(figsize = (30, 15))
for i in range(1, 7, ):
    plt.subplot(1, 7, i)
    plt.imshow(x[i])
    plt.axis("off")
plt.show()

# Pre-processing Stage

In [ ]:
# We split LESIONS (moles), not images. Many lesions have several photos. If photos of the
# same lesion land in both train and test, the model is graded on lesions it already saw.

# Step 1: one row per lesion, with its diagnosis (every lesion has exactly one)
lesions = df.drop_duplicates("lesion_id")[["lesion_id", "lesion_ID"]]

# Step 2: split the LESIONS 80% train / 10% validation / 10% test.
# stratify keeps the mix of diagnoses similar in every part.
train_les, rest_les = train_test_split(lesions, test_size=0.20, random_state=28,
                                       stratify=lesions["lesion_ID"])
val_les, test_les   = train_test_split(rest_les, test_size=0.50, random_state=28,
                                       stratify=rest_les["lesion_ID"])

# Step 3: every image goes to the same split as its lesion.
# is_train is a True/False value per image; x[is_train] keeps only the True rows.
is_train = df["lesion_id"].isin(train_les["lesion_id"]).values
is_val   = df["lesion_id"].isin(val_les["lesion_id"]).values
is_test  = df["lesion_id"].isin(test_les["lesion_id"]).values

x_train, y_train = x[is_train], y[is_train]
x_val,   y_val   = x[is_val],   y[is_val]
x_test,  y_test  = x[is_test],  y[is_test]


In [ ]:
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

Both splits above group by `lesion_id`, so all photos of one lesion stay together (the asserts in the split cell check this).

# 1. Imports and device

`device` is where the maths happens. A GPU is hundreds of times faster here, but PyTorch does not use it automatically - both the model and every batch have to be moved there by hand.

If this prints `cpu`, go to **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score)

torch.manual_seed(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("No GPU! Runtime > Change runtime type > T4 GPU, or this takes hours.")

# 2. Dataset and DataLoader

A **Dataset** answers two questions: how many items are there, and give me item `i`. A **DataLoader** wraps it and hands over *batches* of 64 images, which is what makes GPU training fast.

**Augmentation** makes small random changes to each training image every epoch - flips, rotations, slight colour shifts - so the model never sees exactly the same image twice. A lesion has no correct orientation, so flips and rotations are safe. Colour jitter stays mild because colour is genuinely diagnostic in dermatoscopy.

Validation and test images are **never** augmented. They have to stay fixed to be a fair measurement.

In [ ]:
# Normalization values from the TRAINING set only - using all the data would leak
# test information into training.
# Images are uint8 (0-255); the Dataset below rescales them to floats in [0, 1], so the
# statistics must be on that same 0-1 scale (hence the / 255).
MEAN = (x_train.mean(axis=(0, 1, 2)) / 255).astype(np.float32)
STD = (x_train.std(axis=(0, 1, 2)) / 255).astype(np.float32)
print("channel means:", MEAN.round(3), "| stds:", STD.round(3))

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.Normalize(MEAN, STD),
])

eval_tf = transforms.Normalize(MEAN, STD)   # no augmentation: measuring, not training


class SkinDataset(Dataset):
    def __init__(self, x, y, tf=None):
        self.x, self.y, self.tf = x, y, tf

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        # OpenCV gave (height, width, channels); PyTorch wants (channels, height, width).
        # uint8 -> float in [0, 1]: Normalize rejects integer tensors, and ColorJitter
        # expects floats in [0, 1].
        img = torch.from_numpy(self.x[i]).permute(2, 0, 1).float().div(255)
        if self.tf is not None:
            img = self.tf(img)
        return img, int(self.y[i])


BATCH_SIZE = 64

# shuffle=True on training only: the CSV is sorted by diagnosis, so without it the
# first batches would be all one class.
# drop_last=True throws away a final undersized batch - BatchNorm misbehaves on 1-2 images.
train_loader = DataLoader(SkinDataset(x_train, y_train, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader   = DataLoader(SkinDataset(x_val, y_val, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(SkinDataset(x_test, y_test, eval_tf),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"{len(train_loader)} training batches of {BATCH_SIZE}")

In [ ]:
# Look at the augmentation before training on it.
ds = SkinDataset(x_train, y_train, train_tf)

def unnormalize(t):
    return np.clip(t.permute(1, 2, 0).numpy() * STD + MEAN, 0, 1)

plt.figure(figsize=(18, 3))
plt.subplot(1, 7, 1); plt.imshow(x_train[0]); plt.title("original"); plt.axis("off")
for k in range(2, 8):
    plt.subplot(1, 7, k); plt.imshow(unnormalize(ds[0][0]))
    plt.title("augmented"); plt.axis("off")
plt.tight_layout(); plt.show()

# 3. The CNN

Four blocks. Each one: two convolutions that look for patterns, BatchNorm to keep the numbers in a stable range, ReLU so the network can model non-linear things, then MaxPool to halve the image.

The image shrinks 71 → 35 → 17 → 8 → 4 while channels grow 32 → 64 → 128 → 256: less spatial detail, more pattern types.

The head uses **global average pooling** rather than `Flatten` + a big dense layer. Flatten+Dense(512) here would cost 2.1M parameters; this costs 1,799. With only ~8,000 training images those extra parameters mostly memorise the training set.

In [ ]:
"""SkinCNN written in the lecture's explicit __init__ / forward style.

Every layer is named in __init__; forward() spells out the order of operations.
This replaces the nn.Sequential block() helper in cell 36 of the notebook.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class CNN(nn.Module):
    def __init__(self, num_classes=7, dropout=0.4):
        super(CNN, self).__init__()

        # ---- block 1: 128 -> 64 ----
        self.conv1a = nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False)
        self.bn1b = nn.BatchNorm2d(32)

        # ---- block 2: 64 -> 32 ----
        self.conv2a = nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False)
        self.bn2b = nn.BatchNorm2d(64)

        # ---- block 3: 32 -> 16 ----
        self.conv3a = nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False)
        self.bn3a = nn.BatchNorm2d(128)
        self.conv3b = nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False)
        self.bn3b = nn.BatchNorm2d(128)

        # ---- block 4: 16 -> 8 ----
        self.conv4a = nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False)
        self.bn4a = nn.BatchNorm2d(256)
        self.conv4b = nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False)
        self.bn4b = nn.BatchNorm2d(256)

        # ---- block 5: 8 -> 4 ----
        self.conv5a = nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False)
        self.bn5a = nn.BatchNorm2d(512)
        self.conv5b = nn.Conv2d(512, 512, kernel_size=3, padding=1, bias=False)
        self.bn5b = nn.BatchNorm2d(512)

        # ---- head ----
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(p=dropout)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, img):
        # block 1
        activation = F.relu(self.bn1a(self.conv1a(img)))
        activation = F.relu(self.bn1b(self.conv1b(activation)))
        activation = self.pool(activation)

        # block 2
        activation = F.relu(self.bn2a(self.conv2a(activation)))
        activation = F.relu(self.bn2b(self.conv2b(activation)))
        activation = self.pool(activation)

        # block 3
        activation = F.relu(self.bn3a(self.conv3a(activation)))
        activation = F.relu(self.bn3b(self.conv3b(activation)))
        activation = self.pool(activation)

        # block 4
        activation = F.relu(self.bn4a(self.conv4a(activation)))
        activation = F.relu(self.bn4b(self.conv4b(activation)))
        activation = self.pool(activation)

        # block 5
        activation = F.relu(self.bn5a(self.conv5a(activation)))
        activation = F.relu(self.bn5b(self.conv5b(activation)))
        activation = self.pool(activation)

        # global average pooling -> (batch, 512, 1, 1) -> (batch, 512)
        pooled = self.gap(activation)
        flattened = pooled.view(-1, 512)

        # No softmax here, for the same reason the slide has no sigmoid:
        # nn.CrossEntropyLoss applies it internally and needs raw logits.
        return self.fc(self.drop(flattened))


model = CNN(num_classes=7).to(device)
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# Check the shapes before training. Run this any time you change the architecture -
# a mismatch shows up here in two seconds instead of halfway through an epoch.
dummy = torch.randn(4, 3, IMG_SIZE, IMG_SIZE).to(device)
out = model(dummy)
print("output shape:", tuple(out.shape), " <- (images, class scores)")

h = dummy
for i, b in enumerate(model.features, 1):
    h = b(h)
    print(f"  after block {i}: {tuple(h.shape)}")
print(f"  after global average pooling: {tuple(model.pool(h).shape)}")

# Note: these are raw scores (logits), NOT probabilities. nn.CrossEntropyLoss
# applies softmax itself - adding one here would quietly weaken training.

# 4. Loss, optimizer, schedule

**The loss** is one number saying how wrong the model is; training minimises it. `CrossEntropyLoss` takes raw scores and *integer* labels - which is why this model has no softmax and why you never one-hot encode in PyTorch.

**Class weights** handle the imbalance. `nv` is 67% of the data, so an unweighted model can score well by almost always guessing it. Weighting makes rare-class mistakes cost more.

**The optimizer** does the learning. **The scheduler** lowers the learning rate along a cosine curve: big steps early, small steps late.

In [ ]:
# bincount counts BY INDEX, so the order matches lesion_ID automatically.
counts = np.bincount(y_train, minlength=7)

# CHANGED: sqrt of the inverse frequency instead of the raw ratio.
# Raw inverse frequency gave nv a weight of 0.05 against df's 2.67 - a 53x spread.
# That told the model a nevus mistake was nearly free, so it sprayed rare-class
# predictions: 156 of 669 test nevi came back as melanoma and melanoma precision
# fell to 0.29. The sqrt keeps the ordering but compresses the spread to ~7.6x.
weights = np.sqrt(len(y_train) / (7 * np.maximum(counts, 1)))
weights = weights / weights.mean()      # keep the average weight at 1
weights = torch.tensor(weights, dtype=torch.float32, device=device)

for i in range(7):
    print(f"{names[i]:<32} {counts[i]:>5} images   weight {weights[i]:.2f}")

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

EPOCHS = 60           # CHANGED: 30 was not enough - val score was still rising at the end
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


# 5. The training loop

The five numbered lines inside the batch loop are the entire learning process; everything else is bookkeeping.

1. **`zero_grad()`** - PyTorch *accumulates* gradients, so they must be cleared. Forgetting this is the most common PyTorch bug.
2. **Forward** - run images through the network.
3. **Loss** - compare the scores to the true labels.
4. **`backward()`** - work out which direction each of the 1.18M weights should move.
5. **`step()`** - move them.

`model.train()` vs `model.eval()` is not optional: dropout must be off when measuring, and BatchNorm has to use its running statistics instead of the current batch's.

The loop keeps the **best** weights, not the last epoch's, and judges "best" by **balanced accuracy** - plain accuracy would happily pick a model that ignores dermatofibroma entirely.

In [ ]:
@torch.no_grad()      # no gradients needed when measuring: saves memory and time
def evaluate(model, loader):
    model.eval()
    total_loss, preds, trues = 0.0, [], []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        total_loss += criterion(out, yb).item() * yb.size(0)
        preds.append(out.argmax(1).cpu())
        trues.append(yb.cpu())
    preds = torch.cat(preds).numpy()
    trues = torch.cat(trues).numpy()
    return total_loss / len(trues), preds, trues


history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_bacc": [], "val_f1": []}
best_f1, best_state, patience, since_best = -1.0, None, 12, 0

for epoch in range(EPOCHS):
    model.train()
    running, seen = 0.0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()          # 1. clear gradients from the last batch
        out = model(xb)                # 2. forward: predictions
        loss = criterion(out, yb)      # 3. how wrong they are
        loss.backward()                # 4. backward: gradients
        optimizer.step()               # 5. nudge the weights

        running += loss.item() * yb.size(0)
        seen += yb.size(0)

    scheduler.step()

    train_loss = running / seen
    val_loss, val_preds, val_true = evaluate(model, val_loader)
    acc = accuracy_score(val_true, val_preds)
    bacc = balanced_accuracy_score(val_true, val_preds)
    f1 = f1_score(val_true, val_preds, average='macro')

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(acc)
    history["val_bacc"].append(bacc)
    history["val_f1"].append(f1)

    flag = ""
    # CHANGED: select on macro F1. Balanced accuracy is the mean of per-class
    # RECALL and contains no notion of precision, so it happily keeps a model
    # that buys rare-class recall with a flood of false positives.
    if f1 > best_f1:
        best_f1, since_best = f1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        flag = "  <- best so far"
    else:
        since_best += 1

    print(f"epoch {epoch+1:>2}/{EPOCHS} | train loss {train_loss:.4f} | "
          f"val loss {val_loss:.4f} | acc {acc:.4f} | bacc {bacc:.4f} | macroF1 {f1:.4f}{flag}")

    if since_best >= patience:
        print(f"stopping early - no improvement for {patience} epochs")
        break

model.load_state_dict(best_state)       # go back to the best weights
print(f"\nbest validation macro F1: {best_f1:.4f}")

# 6. Curves and final evaluation

Read the loss plot carefully. Both lines falling together is healthy. Training loss falling while **validation loss rises** is overfitting - the model is memorising. The fix is more dropout, stronger augmentation, or a smaller model.

The test cell runs **once**, at the very end. That number is the one for your report.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

ax[0].plot(history["train_loss"], label="train")
ax[0].plot(history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()

ax[1].plot(history["val_acc"], label="accuracy")
ax[1].plot(history["val_bacc"], label="balanced accuracy")
ax[1].set_title("Validation performance"); ax[1].set_xlabel("Epoch"); ax[1].legend()

plt.tight_layout(); plt.show()

In [ ]:
test_loss, y_pred, y_true = evaluate(model, test_loader)

print(f"test loss        : {test_loss:.4f}")
print(f"accuracy         : {accuracy_score(y_true, y_pred):.4f}")
print(f"balanced accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=names, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(1, 2, figsize=(20, 7))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=names, yticklabels=names, ax=ax[0])
ax[0].set_title("Confusion matrix (counts)", fontsize=16)
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, cbar=False,
            xticklabels=names, yticklabels=names, ax=ax[1])
ax[1].set_title("Normalized by true class (diagonal = recall)", fontsize=16)
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True")

plt.tight_layout(); plt.show()

# Each ROW is a true class, showing where those images actually went.
# The melanoma row is the one to discuss: melanoma predicted as nevus is the
# clinically costly error.

In [ ]:
per_class = pd.DataFrame({
    "Lesion": names,
    "Recall": cm_norm.diagonal().round(3),
    "Test images": cm.sum(axis=1),
})

bars = plt.bar(per_class["Lesion"], per_class["Recall"])
plt.bar_label(bars, fmt="%.2f")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.title("Accuracy per Lesion Type")
plt.ylabel("Recall")
plt.tight_layout(); plt.show()

per_class

In [ ]:
# Save the trained model so a runtime restart does not cost you the training run.
torch.save({
    "state_dict": model.state_dict(),
    "classes": names,
    "img_size": 71,
    "mean": MEAN, "std": STD,
}, "/content/drive/MyDrive/ML Project/skin_cnn.pt")

print("saved to Drive")

# 7. Measured results

Both models were trained on the full HAM10000 set (9,958 images with a recorded age),
split 80/10/10 **by `lesion_id`** so no physical lesion appears in two splits. The test
set is 1,010 images from 742 held-out lesions. Identical split, identical seed, so the
two rows are directly comparable.

| Metric | Original notebook | This version | Change |
|---|---|---|---|
| Accuracy | 0.592 | **0.717** | +12.5 |
| Balanced accuracy | 0.643 | 0.629 | −1.4 |
| Macro F1 | 0.461 | **0.584** | +12.3 |
| Weighted F1 | 0.636 | **0.738** | +10.2 |
| Cohen's kappa | 0.404 | **0.529** | +12.5 |
| Macro ROC-AUC | 0.893 | **0.939** | +4.6 |
| Malignant sensitivity | 0.823 | **0.847** | +2.4 |
| Malignant specificity | 0.677 | **0.786** | +10.9 |
| Melanomas missed as benign | 27 / 116 | **25 / 116** | −2 |

Cohen's kappa is the honest headline: it corrects for chance agreement, so moving
0.404 → 0.529 shows the gain is real signal rather than the model simply learning to
answer "nevus" more often.

**The trade-off, stated plainly.** Balanced accuracy fell 1.4 points. That is the direct
consequence of softening the class weights — majority-class precision is bought with a
little rare-class recall. Dermatofibroma is where it shows (recall 0.538 → 0.308), though
that is 7 vs 4 images out of 13 and is noise at that sample size.

## Per-class, final model

| Lesion | Precision | Recall | F1 | Test images |
|---|---|---|---|---|
| Melanocytic nevi | 0.941 | 0.765 | 0.844 | 669 |
| Melanoma | 0.362 | 0.690 | 0.475 | 116 |
| Benign keratosis | 0.561 | 0.495 | 0.526 | 111 |
| Basal cell carcinoma | 0.562 | 0.745 | 0.641 | 55 |
| Actinic keratoses | 0.431 | 0.688 | 0.530 | 32 |
| Vascular lesions | 0.667 | 0.714 | 0.690 | 14 |
| Dermatofibroma | 0.500 | 0.308 | 0.381 | 13 |

## What produced the gain

| Change | Cell | Why |
|---|---|---|
| 71×71 → 128×128 input | 22 | Source images are 600×450; 71×71 destroys the diagnostic texture |
| 4 blocks → 5, 256 → 512 channels | 36 | A 128px input needs one more downsampling stage |
| Class weights: 53× spread → 7.6× | 39 | Raw inverse frequency inverted the imbalance instead of correcting it |
| Select on macro F1, not balanced accuracy | 41 | Balanced accuracy ignores precision entirely |
| 30 → 60 epochs | 39 | The original run never converged; early stopping never fired |
| Test-time augmentation + per-lesion pooling | below | +0.3 and +2.1 accuracy points, no retraining |


In [ ]:
# Test-time augmentation: a mole has no correct orientation, so average the
# prediction over all four flips.
@torch.no_grad()
def predict_tta(model, loader):
    model.eval()
    probs, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        p = model(xb).softmax(1)
        for flip in (lambda t: t.flip(-1), lambda t: t.flip(-2),
                     lambda t: t.flip(-1).flip(-2)):
            p = p + model(flip(xb)).softmax(1)
        probs.append((p / 4).cpu())
        trues.append(yb)
    return torch.cat(probs).numpy(), torch.cat(trues).numpy()


prob, y_true = predict_tta(model, test_loader)
print(f"accuracy with TTA: {accuracy_score(y_true, prob.argmax(1)):.4f}")


In [ ]:
# Per-lesion pooling: HAM10000 photographs many lesions more than once, and the
# split already keeps those photos together. Averaging a lesion's photos into one
# prediction is both a real gain and a better match to how a clinician works.
test_lesions = df.loc[is_test, "lesion_id"].values

prob_pooled = prob.copy()
for lid in np.unique(test_lesions):
    m = test_lesions == lid
    prob_pooled[m] = prob[m].mean(axis=0)

print(f"accuracy with TTA + pooling: {accuracy_score(y_true, prob_pooled.argmax(1)):.4f}")
print()
print(classification_report(y_true, prob_pooled.argmax(1), target_names=names,
                            digits=3, zero_division=0))
